# Checkpoint 01 - primeira leitura do Firebase

Este checkpoint parte do ambiente já verificado no **Checkpoint 00**.

A novidade é acessar o **Firebase Realtime Database pela API REST**, conhecer a estrutura do pacote e validar os vetores recebidos.

## 1. Configuração

A URL e o caminho do nó serão reutilizados nos próximos checkpoints.

In [ ]:
# CONFIGURAÇÃO DA OFICINA
# Use a mesma URL do Firebase Realtime Database fornecida para a oficina.
FIREBASE_URL = "https://oficina-python-saude-teste-default-rtdb.firebaseio.com"  # Endereço base do Realtime Database usado na oficina.
STREAM_PATH = "oficina/live"  # Caminho do nó que contém sempre o pacote mais recente.

# Parâmetros da oficina
FS = 250  # Frequência de amostragem dos sinais: 250 amostras por segundo.
AMOSTRAS_POR_PACOTE = 50  # Quantidade esperada de amostras em cada pacote publicado.
JANELA_S = 8  # Duração, em segundos, da janela de sinal mantida no buffer.

## 2. Reutilização do Checkpoint 00

Continuamos usando `requests` para comunicação HTTP e `NumPy` para trabalhar com os vetores numéricos.
Este bloco não introduz um conceito novo da oficina.

In [ ]:
import requests                     # Faz a leitura HTTP do pacote atual armazenado no Firebase.
import numpy as np                  # Converte as listas recebidas em vetores numéricos para validação e processamento.

## 3. Novidade deste checkpoint — primeira leitura REST

Agora acrescentamos:

- a montagem da URL REST do nó;
- uma leitura HTTP `GET`;
- a conversão da resposta JSON para Python;
- a validação da estrutura e dos tamanhos do pacote.

In [ ]:
# Função: Monta a URL REST completa do nó do Firebase, evitando barras duplicadas.
def firebase_node_url(firebase_url, node=STREAM_PATH):
    base = firebase_url.rstrip("/")  # Remove uma barra final para evitar "//" ao montar a URL.
    node = node.strip("/")  # Remove barras no início/fim do caminho antes de juntá-lo à URL.
    return f"{base}/{node}.json"  # O sufixo .json permite acessar esse nó pela API REST do Firebase.

# Função: Faz uma leitura única do pacote atualmente armazenado no nó da oficina.
def ler_pacote_atual(firebase_url=FIREBASE_URL, node=STREAM_PATH):
    r = requests.get(firebase_node_url(firebase_url, node), timeout=10)  # Faz um GET e espera no máximo 10 s pela resposta.
    r.raise_for_status()  # Interrompe com erro se o servidor responder com código HTTP de falha.
    return r.json()  # Converte a resposta JSON do Firebase diretamente em um objeto Python.

# Função: Confere os campos e tamanhos do pacote e converte os sinais para vetores NumPy.
def validar_pacote(pacote):
    campos = (
        "session_id", "seq", "sample_start", "fs", "n",
        "ecg", "ppg_ir", "ppg_red"
    )
    faltando = [k for k in campos if k not in pacote]  # Lista quais campos obrigatórios não chegaram no pacote.
    if faltando:
        raise ValueError(f"campos ausentes: {faltando}")

    n = int(pacote["n"])  # Converte o tamanho informado para inteiro antes de comparar com os vetores.
    ecg = np.asarray(pacote["ecg"], dtype=float)  # Converte a lista de ECG recebida em vetor numérico.
    ir = np.asarray(pacote["ppg_ir"], dtype=float)  # Converte o canal PPG infravermelho em vetor numérico.
    red = np.asarray(pacote["ppg_red"], dtype=float)  # Converte o canal PPG vermelho em vetor numérico.

    if not (len(ecg) == len(ir) == len(red) == n):
        raise ValueError("vetores com tamanhos incompatíveis")

    return {
        "session_id": str(pacote["session_id"]),
        "seq": int(pacote["seq"]),
        "sample_start": int(pacote["sample_start"]),
        "fs": float(pacote["fs"]),
        "n": n,
        "timestamp_pub": float(pacote.get("timestamp_pub", 0.0)),
        "ecg": ecg,
        "ppg_ir": ir,
        "ppg_red": red,
    }

## 4. Executar e inspecionar o primeiro pacote

Esta célula usa as funções definidas acima e mostra os principais campos recebidos.

In [ ]:
pacote = ler_pacote_atual()
p = validar_pacote(pacote)

print("session_id:", p["session_id"])
print("seq:", p["seq"])
print("sample_start:", p["sample_start"])
print("fs:", p["fs"], "Hz")
print("n:", p["n"])
print("ECG (5 primeiras):", p["ecg"][:5])
print("PPG IR (5 primeiras):", p["ppg_ir"][:5])
print("PPG Red (5 primeiras):", p["ppg_red"][:5])
print("CHECKPOINT 1 OK")